# فاز اول — مسیر ۲ (Classic ML) | EDA + آماده‌سازی داده + Baseline

تو این نوت‌بوک من کارهای فاز اول رو یکجا و قابل اجرا جمع کردم:

1) تعریف مسئله و نگاه کلی به داده
2) EDA (حداقل ۶ نمودار)
3) آماده‌سازی داده (Split اصولی + Preprocess Pipeline)
4) Baseline Logistic Regression + Confusion Matrix + ROC

> پیش‌نیاز: فایل CSV در مسیر `data/raw/bank_marketing.csv` باشد (یا دانلود خودکار را اجرا کنید).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.load_bank_marketing import load_raw, split_xy
from src.training.prepare_data import to_binary_target

df = load_raw()
X, y_raw, target_col = split_xy(df, target_col=None)
y, y_info = to_binary_target(y_raw)

print('Data shape:', df.shape)
print('Detected target column:', target_col)
print('Target mapping:', y_info.get('mapping'))
print('Unique labels (raw):', y_info.get('unique_labels'))
print('Numeric unique (raw):', y_info.get('numeric_unique'))
df.head()

## 1) بررسی کیفیت داده و نوع ستون‌ها


In [ ]:
df.info()

In [ ]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
print('num_cols:', len(num_cols))
print('cat_cols:', len(cat_cols))
num_cols[:10], cat_cols[:10]

## 2) EDA — توزیع برچسب و عدم‌توازن کلاس


In [ ]:
vc = pd.Series(y).value_counts(dropna=False)
vc

In [ ]:
plt.figure()
vc.plot(kind='bar')
plt.title('Plot 1 — Target distribution (after binarization)')
plt.xlabel('class')
plt.ylabel('count')
plt.tight_layout()
plt.show()

if len(vc) < 2:
    print('⚠️ هشدار: برچسب فقط یک کلاس دارد. این یعنی دیتاست/ستون هدف اشتباه است یا مپینگ اشتباه انجام شده.')

## 3) EDA — Missing / Unknown

در بعضی دیتاست‌های صنعتی مقدارهایی مثل `unknown` عملاً نقش Missing رو دارند. اینجا هم دو حالت رو چک می‌کنم:
- NaN واقعی
- رشته‌ی unknown (اگر ستون‌های دسته‌ای داشته باشیم)


In [ ]:
nan_rate = X.isna().mean().sort_values(ascending=False)
nan_rate.head(20)

In [ ]:
plt.figure(figsize=(8,4))
nan_rate.head(15).plot(kind='bar')
plt.title('Plot 2 — Top columns by NaN rate')
plt.ylabel('rate')
plt.tight_layout()
plt.show()

In [ ]:
if len(cat_cols) > 0:
    unknown_rate = {}
    for c in cat_cols:
        unknown_rate[c] = (X[c].astype(str).str.lower() == 'unknown').mean()
    unknown_rate = pd.Series(unknown_rate).sort_values(ascending=False)
    display(unknown_rate.head(15))

    plt.figure(figsize=(8,4))
    unknown_rate.head(12).plot(kind='bar')
    plt.title('Plot 3 — Top categorical columns by UNKNOWN rate')
    plt.ylabel('rate')
    plt.tight_layout()
    plt.show()
else:
    print('این دیتاست ستون دسته‌ای ندارد یا همه ستون‌ها عددی هستند.')

## 4) EDA — توزیع ویژگی‌های عددی

هدف: دیدن skew/outlier و اینکه آیا scaling / transform نیاز داریم یا نه.


In [ ]:
cols = num_cols[:6] if len(num_cols) >= 6 else num_cols
if len(cols) > 0:
    X[cols].hist(bins=30, figsize=(10,6))
    plt.suptitle('Plot 4 — Numeric feature histograms')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric columns found!')

## 5) EDA — جداشدگی ویژگی‌ها بین کلاس‌ها (Boxplot)


In [ ]:
box_cols = num_cols[:4]
if len(box_cols) > 0 and len(np.unique(y)) == 2:
    tmp = df[box_cols].copy()
    tmp['y_bin'] = y
    tmp.boxplot(column=box_cols, by='y_bin', figsize=(10,4))
    plt.suptitle('')
    plt.title('Plot 5 — Numeric features by class (boxplot)')
    plt.tight_layout()
    plt.show()
else:
    print('Boxplot نیاز به حداقل ۲ کلاس و چند ستون عددی دارد.')

## 6) EDA — همبستگی ویژگی‌های عددی (Correlation)


In [ ]:
if len(num_cols) > 1:
    corr = df[num_cols].corr(numeric_only=True)
    plt.figure(figsize=(8,6))
    plt.imshow(corr, aspect='auto')
    plt.colorbar()
    plt.title('Plot 6 — Correlation matrix (numeric features)')
    plt.tight_layout()
    plt.show()
else:
    print('برای correlation حداقل ۲ ستون عددی لازم است.')

## 7) EDA — یک نمای دوبعدی (Scatter)


In [ ]:
if len(num_cols) >= 2 and len(np.unique(y)) == 2:
    x1, x2 = num_cols[0], num_cols[1]
    plt.figure(figsize=(6,4))
    for cls, mk in [(0, 'o'), (1, 'x')]:
        sub = df[pd.Series(y, index=df.index) == cls]
        plt.scatter(sub[x1], sub[x2], s=10, alpha=0.4, marker=mk, label=str(cls))
    plt.xlabel(x1)
    plt.ylabel(x2)
    plt.title(f'Plot 7 — Scatter: {x1} vs {x2} by class')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Scatter plot نیاز به حداقل ۲ ستون عددی و ۲ کلاس دارد.')

## 8) آماده‌سازی داده — Split اصولی + Preprocess Pipeline

اینجا از pipeline ماژولار استفاده می‌کنم:
- پاکسازی (unknown→missing)
- Feature Engineering (در صورت وجود ستون‌های مرتبط)
- imputation + scaling برای عددی‌ها
- one-hot برای categorical ها

Split هم به شکل stratified انجام می‌شود (train/valid/test).


In [ ]:
from src.training.split import stratified_split
from src.features.preprocess import build_preprocess_pipeline

split = stratified_split(X, pd.Series(y), seed=42, test_size=0.2, valid_size=0.2)
print('train/valid/test sizes:', len(split.y_train), len(split.y_valid), len(split.y_test))
print('pos_rate:', float(np.mean(split.y_train)), float(np.mean(split.y_valid)), float(np.mean(split.y_test)))

preprocess = build_preprocess_pipeline(drop_duration=True)
Xtr = preprocess.fit_transform(split.X_train)
Xva = preprocess.transform(split.X_valid)
Xte = preprocess.transform(split.X_test)

print('X shapes:', Xtr.shape, Xva.shape, Xte.shape)
try:
    feat_names = preprocess.named_steps['preprocess'].get_feature_names_out()
    print('n_features:', len(feat_names))
except Exception:
    feat_names = None
    print('feature names not available')

## 9) Baseline — Logistic Regression + ارزیابی

مدل baseline رو Logistic Regression انتخاب کردم چون:
- سریع و قابل تفسیر است
- نقطه‌ی شروع خوبی برای مقایسه با مدل‌های پیچیده‌تر است
- با regularization و class_weight می‌تواند برای داده‌های نامتوازن هم مناسب باشد


In [ ]:
from src.models.baselines import BaselineConfig, build_logreg
from src.evaluation.metrics import compute_classification_report, report_to_dict
from src.evaluation.plots import save_confusion_matrix, save_roc_curve
from pathlib import Path

model = build_logreg(BaselineConfig(seed=42))
model.fit(Xtr, split.y_train)

def evaluate(name, Xs, ys):
    prob = model.predict_proba(Xs)[:, 1]
    pred = (prob >= 0.5).astype(int)
    rep = compute_classification_report(ys, pred, prob)
    print(name, report_to_dict(rep))
    return prob, pred, rep

prob_va, pred_va, rep_va = evaluate('valid', Xva, split.y_valid)
prob_te, pred_te, rep_te = evaluate('test', Xte, split.y_test)

out_dir = Path('results')
fig_dir = out_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
save_confusion_matrix(split.y_valid, pred_va, fig_dir / 'phase1_cm_valid.png', 'Phase-1 Baseline — CM (Valid)')
save_roc_curve(split.y_valid, prob_va, fig_dir / 'phase1_roc_valid.png', 'Phase-1 Baseline — ROC (Valid)')

save_confusion_matrix(split.y_test, pred_te, fig_dir / 'phase1_cm_test.png', 'Phase-1 Baseline — CM (Test)')
save_roc_curve(split.y_test, prob_te, fig_dir / 'phase1_roc_test.png', 'Phase-1 Baseline — ROC (Test)')

print('Saved plots to:', fig_dir)

## 10) جمع‌بندی فاز اول و پلن فاز دوم

در فاز اول:
- داده و ستون هدف بررسی شد
- EDA انجام شد و حداقل ۶ نمودار رسم شد
- Split اصولی و pipeline پیش‌پردازش پیاده‌سازی شد
- یک مدل baseline (Logistic Regression) آموزش و ارزیابی شد

در فاز دوم:
- مدل‌های SVM / KNN / RandomForest / XGBoost را هم پیاده‌سازی می‌کنم
- با جستجوی ابرپارامترها (Grid/Random) نتایج را بهبود می‌دم
- برای مدل‌های درختی Feature Importance گزارش می‌کنم
- یک دمو (Streamlit) می‌سازم تا ورودی بگیره و پیش‌بینی/احتمال رو نمایش بده
